在 transformers 库中，我们通常使用 AutoModelForCausalLM 和 AutoTokenizer 这两个类来自动加载与模型匹配的权重和分词器。下面这段代码会自动从 Hugging Face Hub 下载所需的模型文件和分词器配置，这可能需要一些时间，具体取决于你的网络速度。

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 指定模型ID
model_id = "Qwen/Qwen1.5-0.5B-Chat" # Chat：经过对话或指令调优的版本。

# 设置设备，优先使用GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_id) # 这一步会根据 model_id 自动下载并加载 Qwen 对应的分词器。

# 加载因果语言模型（CasualLanguageModel, CLM, Qwen1.5 Chat 属于 Decoder-only 的因果语言模型），并将其移动到指定设备
model = AutoModelForCausalLM.from_pretrained(model_id).to(device) # 从 Hugging Face 下载模型配置和权重，并创建模型对象。
# (模型和输入张量必须位于同一个设备上)
print("模型和分词器加载完成！")

Using device: cuda


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

模型和分词器加载完成！


我们来创建一个对话提示，Qwen1.5-Chat 模型遵循特定的对话模板。然后，可以使用上一步加载的 tokenizer 将文本提示转换为模型能够理解的数字 ID（即 Token ID）。

In [7]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "你好， 请介绍一下自己。"
    }
]

# 使用分词器的模板格式化输入
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,  # 表示这里只进行“聊天格式转换”，暂时不进行 Token 编码。
    add_generation_prompt = True  # 表示在格式化文本末尾添加“轮到助手回答”的提示标记。
)

# 编码输入文本(这一步才真正把文本转换成模型能够处理的张量。)
model_inputs = tokenizer([text], return_tensors='pt').to(device) # [text]，方括号表示是一个batch， 如果有两个文本可以写成 [text1, text2]
# return_tensors='pt' 表示返回的是pytorch张量 ，其中 pt = PyTorch， 如果不写这一项，通常返回普通 Python 列表。
# 分词器默认生成的张量位于 CPU， 而模型已经在GPU, 所以输入也必须移动到GPU



print('编码后的输入文本：')
print(model_inputs)

# {
#     'input_ids': tensor(  保存每个token对应的整数编号，形状（B,L）B：batch_size, L:序列token数量
#     	[
#     		[151644,   8948,    198,   2610,    525,    264,  10950,   1071,  50944, 13, 151645,    198, 151644,    872,    198, 108386,   				3837,    220, 14880, 109432,  99283,   1773, 151645,    198, 151644,  77091,    198]
#         ], device='cuda:0'), 
# 
#      # mask表示告诉模型哪些位置是真实的token，哪些位置是为了对齐长度而加入的Padding,其中1表示有效，0表示无效（当前只有一个句子，所有没有补齐内容）
#     'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,1, 1, 1]], device='cuda:0')
# }
print(model_inputs["input_ids"].shape)

编码后的输入文本：
{'input_ids': tensor([[151644,   8948,    198,   2610,    525,    264,  10950,  17847,     13,
         151645,    198, 151644,    872,    198, 108386,   3837,    220,  14880,
         109432,  99283,   1773, 151645,    198, 151644,  77091,    198]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1]], device='cuda:0')}
torch.Size([1, 26])


现在可以调用模型的 generate() 方法来生成回答了。模型会输出一系列 Token ID，这代表了它的回答。

最后，我们需要使用分词器的 decode() 方法，将这些数字 ID 翻译回人类可以阅读的文本。

In [8]:
# 使用模型生成回答
# max_new_token 控制了模型最多能够生成多少个新的token
generated_ids = model.generate(model_inputs.input_ids, max_new_tokens = 1024)


# 将生成的 Token ID 截取掉输入部分
# 这样我们只解码模型重新生成的部分
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

# 解码生成的Token ID
response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

print("\n 模型生成的回答：")
print(response)


 模型生成的回答：
您好！我是来自阿里云的超大规模语言模型“通义千问”。我是一个基于多模态、多训练的数据驱动的语言模型，可以回答问题、创作文字，还能表达观点、撰写代码。有什么我可以帮助您的吗？
